In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import kagglehub

path = kagglehub.dataset_download(r'user2036/fruit-freshness-dataset-v1')

100%|██████████| 1.48G/1.48G [01:37<00:00, 16.3MB/s]

Extracting files...


## **Dataset Loading, Splitting, and Class Extraction**

In this step, the image dataset is loaded from the directory and automatically split into training and validation sets using TensorFlow utilities.

### **Dataset Path**
The dataset is organized in a directory structure where:
- Each subfolder represents a separate class.
- Images inside each folder belong to that class.

TensorFlow automatically assigns labels based on these folder names.

### **Image Configuration**
- **Image Size:** All images are resized to 224 × 224 pixels to ensure a fixed input size for the CNN.
- **Batch Size:** Images are processed in batches of 32 to balance training speed and memory usage.

### **Train–Validation Split**
- 80% of the data is used for training.
- 20% of the data is used for validation.
- The same seed value is used for both datasets to ensure a consistent and reproducible split.

### **Input–Output Handling**
The function `image_dataset_from_directory()` returns a TensorFlow dataset where each element is a tuple:



In [3]:
import tensorflow as tf
import os

train_path = os.path.join(path, 'train')
val_path = os.path.join(path, 'val')

In [4]:
image_height = 244
image_width = 244
batch_size = 32

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    subset = 'training',
    seed=123,
    validation_split=0.2)

Found 9453 files belonging to 6 classes.
Using 7563 files for training.


In [6]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    subset = 'validation',
    seed=123,
    validation_split=0.2)

Found 9453 files belonging to 6 classes.
Using 1890 files for validation.


In [7]:
original_class_name_for_display = train_ds.class_names
print(original_class_name_for_display)

['apple', 'banana', 'orange', 'rottenapples', 'rottenbanana', 'rottenoranges']


## Image Normalization for TensorFlow Dataset

Before training a model, it is crucial to preprocess the images to ensure stable and efficient learning. One key preprocessing step is **normalization**.

### What is Normalization?

Normalization scales the pixel values of images from their original range of `[0, 255]` to a smaller range of `[0, 1]`. This is done by dividing each pixel value by 255 and converting it to a `float32` type.  

**Why normalize?**  
- Helps the model converge faster during training.  
- Improves numerical stability.  
- Prevents large pixel values from dominating the gradients.  

### Applying Normalization

Normalization is applied to **both the training and validation datasets** so that the model sees consistently scaled input data in both training and evaluation phases.

### Dataset Information

After normalization, it is important to verify the dataset properties:  
- **Number of batches** in training and validation datasets.  
- **Image dimensions** (height × width).  
- **Batch size** used for training.  
- **Classes present** in the dataset.  

Normalizing the dataset ensures that the model trains efficiently and accurately on properly scaled input images.


In [8]:
def normalize(image, label):
    image = tf.cast(image/255.0, tf.float32)
    return image, label

In [9]:
train_ds = train_ds.map(normalize)
val_ds = val_ds.map(normalize)

## Mapping Original Labels to Binary Classes

After loading and normalizing the dataset, the next step is to **simplify the labels** for binary classification.  

### What is Being Done?

- The original dataset contains multiple classes representing different types of fruits, both **fresh** and **rotten**.  
- For a binary classification task, we map these original labels into **two categories**:  
  - `0` → Fresh fruits (`freshapples`, `freshbanana`, `freshoranges`)  
  - `1` → Rotten fruits (`rottenapples`, `rottenbanana`, `rottenoranges`)  

### How It Works

- The mapping relies on the order of the original class names.  
- Any label belonging to the first half of the class list is considered `fresh` (0), and any label in the second half is considered `rotten` (1).  
- This ensures that the dataset is now suitable for a **binary classification model**, where the model predicts whether a fruit is fresh or rotten.  


## **Casting.**

**Casting** is the process of **converting a data value from one type to another**.

- Example: Turning a logical value `True`/`False` into a number like `1.0` or `0.0`.

## Why Do We Use `tf.cast`?

- TensorFlow models require labels to be in a **specific numerical format** (usually `float32`)
- This is necessary for the model to **calculate errors** and **update weights** during training.

## What Happens Without `tf.cast`?

- The model will raise a **Type Mismatch error**
- Reason: It **cannot perform calculus** (like gradients and backpropagation) on non-numeric logical values.




This step simplifies the problem from multi-class to binary, making it easier to train models specifically for detecting freshness.


In [10]:
def map_binary(image,label):
    binary_label = tf.cast(label >= len(original_class_name_for_display) // 2, tf.float32)
    return image, binary_label

In [11]:
train_ds = train_ds.map(map_binary)
val_ds = val_ds.map(map_binary)

## Data Augmentation for Training Dataset

Data augmentation is a technique used to **artificially increase the diversity of the training dataset**. This helps the model generalize better and reduces overfitting by exposing it to slightly altered versions of the original images.

### What is Being Done?

1. **Augmentation Techniques Applied**  
   - **Random horizontal flipping**: Mirrors the image horizontally with some probability.  
   - **Random rotation**: Rotates the image by a small random angle.  
   - **Random zoom**: Zooms in or out of the image randomly.  

2. **Application to Training Dataset**  
   - The augmentation is applied **only to the training dataset**, not to the validation dataset.  
   - Each image in the training dataset is modified on-the-fly during training, ensuring a diverse set of inputs for the model.  

### Benefits of Data Augmentation

- Increases the effective size of the training dataset without collecting more data.  
- Improves the model’s robustness to variations in input images.  
- Helps prevent overfitting by providing different perspectives of the same data.  

After this step, the training dataset contains augmented images while preserving the corresponding labels, ready for more robust model training.


In [12]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])

In [13]:
def augment(image, label):
    return data_augmentation(image), label

In [14]:
train_ds_augmented = train_ds.map(augment)

## Convolutional Neural Network (CNN) Model Architecture

A Convolutional Neural Network (CNN) is designed to automatically and adaptively learn spatial hierarchies of features from images. This model is structured to classify images into **binary classes (fresh vs rotten fruits)**.

### Architecture Overview

1. **Convolutional Layers (Conv2D)**  
   - Extract feature maps from input images using learnable filters.  
   - The first layer specifies the `input_shape` to match the dataset images.  
   - Subsequent Conv2D layers increase the number of filters (32 → 64 → 128) to capture more complex features.

2. **Pooling Layers (MaxPooling2D)**  
   - Reduce the spatial dimensions of feature maps.  
   - Helps decrease computational load and extract dominant features.  

3. **Flatten Layer**  
   - Converts the 2D feature maps into a 1D vector to feed into fully connected (Dense) layers.  

4. **Dense Layers (Fully Connected Layers)**  
   - Learn high-level representations of the extracted features.  
   - Includes a **Dropout layer** (0.5) for regularization to prevent overfitting.  

5. **Output Layer**  
   - A single neuron with **sigmoid activation** for binary classification (`0 = fresh`, `1 = rotten`).  

### Summary

- The model progressively extracts features from images using convolution and pooling.  
- Fully connected layers interpret these features for classification.  
- Dropout ensures robustness against overfitting.  
- The final sigmoid layer outputs a probability indicating whether the fruit is fresh or rotten.

 This CNN architecture is well-suited for image classification tasks and can be trained effectively on the normalized, augmented dataset.


In [15]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(image_height, image_width, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [16]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 242, 242, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 121, 121, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 119, 119, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 59, 59, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 57, 57, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,938,561 (49.36 MB)

 Trainable params: 12,938,561 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

## Compiling the CNN Model

After defining the CNN architecture, the next step is to **compile the model**, which configures it for training by specifying the **optimizer**, **loss function**, and **metrics**.

### Components of Compilation

1. **Optimizer: Adam**  
   - Adam is a widely used optimization algorithm that combines the benefits of **AdaGrad** and **RMSProp**.  
   - It adapts the learning rate for each parameter, making it efficient and robust for a wide range of problems.

2. **Loss Function: Binary Crossentropy**  
   - Suitable for **binary classification tasks** where the output is either 0 or 1.  
   - Measures the difference between predicted probabilities (from the sigmoid output) and the actual labels.  

3. **Metrics: Accuracy**  
   - Monitors the **percentage of correctly classified examples** during training and evaluation.  
   - Provides a simple, intuitive measure of model performance.

### Summary

- Compiling the model is essential before training.  
- The chosen optimizer, loss function, and metric align perfectly with the binary classification of fresh vs. rotten fruits.  

After this step, the CNN is ready for training on the normalized and augmented dataset.


In [17]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [18]:
Epochs = 20
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

history = model.fit(train_ds_augmented,
                    epochs=Epochs,
                    validation_data=val_ds,
                    callbacks=[early_stopping])

Epoch 1/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 134s 527ms/step - accuracy: 0.8173 - loss: 0.4136 - val_accuracy: 0.8799 - val_loss: 0.2731
Epoch 2/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 120s 507ms/step - accuracy: 0.9068 - loss: 0.2358 - val_accuracy: 0.9296 - val_loss: 0.2016
Epoch 3/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 121s 511ms/step - accuracy: 0.9114 - loss: 0.2179 - val_accuracy: 0.9333 - val_loss: 0.1713
Epoch 4/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 120s 503ms/step - accuracy: 0.9024 - loss: 0.2462 - val_accuracy: 0.9365 - val_loss: 0.1693
Epoch 5/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 120s 505ms/step - accuracy: 0.9178 - loss: 0.2127 - val_accuracy: 0.8698 - val_loss: 0.3007
Epoch 6/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 143s 509ms/step - accuracy: 0.9203 - loss: 0.2051 - val_accuracy: 0.9270 - val_loss: 0.1785
Epoch 7/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 121s 510ms/step - accuracy: 0.9294 - loss: 0.1723 - val_accuracy: 0.9513 - val_loss: 0.1271
Epoch 8/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 120s 504ms/step - accuracy: 0.9405 -

## Training the CNN Model

After compiling the model, the next step is to **train it on the dataset**. Training involves updating the model’s weights to minimize the loss function while improving performance on the validation set.

### Training Configuration

1. **Number of Epochs**  
   - The model is trained for a predefined number of epochs (full passes over the training dataset).  
   - In this case, a maximum of **20 epochs** is set, but early stopping may halt training earlier if performance stabilizes.

2. **EarlyStopping Callback**  
   - Monitors the **validation loss** during training.  
   - If the validation loss does not improve for **5 consecutive epochs**, training stops early to prevent overfitting.  
   - Restores the model weights from the epoch with the **best validation loss**, ensuring optimal performance.

3. **Training on Augmented Data**  
   - The **augmented training dataset** is used so that the model sees a diverse range of inputs, improving generalization.  
   - Validation is performed on the original validation dataset to evaluate true model performance.

### Summary

- Training adjusts the model’s weights using the Adam optimizer and binary crossentropy loss.  
- Early stopping ensures efficient training while preventing overfitting.  
- The training history is stored in a variable, which can later be used for **visualizing loss and accuracy trends**.

After this step, the model is trained and ready for evaluation or deployment.


In [19]:
test_path = os.path.join(path, 'test')

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(image_height, image_width),
    batch_size=batch_size,
    shuffle=False)

Found 1879 files belonging to 6 classes.


In [20]:
test_ds = test_ds.map(normalize)
test_ds = test_ds.map(map_binary)

## Evaluating the Trained CNN Model

After training, it is important to **evaluate the model on unseen data** to understand how well it generalizes.

### Evaluation Process

1. **Test Dataset**  
   - The model is evaluated on the **test dataset**, which contains images that the model has **never seen during training or validation**.  
   - This provides an unbiased estimate of the model’s real-world performance.

2. **Evaluation Metrics**  
   - The `.evaluate()` method computes the **loss** and **accuracy** on the test dataset.  
   - **Loss** indicates how well the model’s predictions match the true labels.  
   - **Accuracy** measures the proportion of correctly classified images (fresh vs rotten).

3. **Storing and Reporting Results**  
   - The test loss and test accuracy are stored in variables for further analysis.  
   - These metrics can be used to compare different models or to visualize performance trends.

### Summary

- Evaluation confirms whether the model has learned meaningful patterns from the training data.  
- High accuracy on the test dataset indicates strong generalization, while a large gap between training and test accuracy may suggest overfitting.

 After this step, the model’s performance on unseen data is quantified and ready for reporting or deployment.


In [21]:
loss, accuracy = model.evaluate(test_ds)

test_loss = loss
test_accuracy = accuracy

59/59 ━━━━━━━━━━━━━━━━━━━━ 11s 179ms/step - accuracy: 0.9808 - loss: 0.0534


In [22]:
print(f'Test Loss: {test_loss}')
print(f'Test Accuracy: {test_accuracy}')

Test Loss: 0.05339831858873367
Test Accuracy: 0.9808408617973328


In [23]:
model_save_path = 'fruit_classification_model.h5'
model.save(model_save_path)

In [24]:
model_save_path = 'fruit_classification_model.keras'
model.save(model_save_path)